In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

from typing import List,Dict
import random
import pandas as pd
import time

/home/hagusta/conda/envs/simple_rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'pandas'

In [2]:
with open("../data/the_republic_introduction.txt") as f:
    doc = f.read()

#chunks = document_to_chunks([doc])

In [20]:
import re

def clean_text_for_chunking(text):
    # 1. Replace 2 or more newlines with a unique placeholder
    # This preserves paragraph structure.
    text = re.sub(r'\n{2,}', '[[PARAGRAPH]]', text)
    
    # 2. Replace single newlines with a space
    # This fixes sentences split across lines (common in PDFs).
    text = text.replace('\n', ' ')
    
    # 3. Restore the paragraph breaks as single newlines
    text = text.replace('[[PARAGRAPH]]', '\n')
    
    # 4. Remove extra spaces created by the process
    text = re.sub(r' +', ' ', text)
    
    return text.strip()

In [37]:
def document_to_chunks (documents:List):
    split_to_chunk = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            model_name='text-embedding-3-small',
            chunk_size = 512,
            chunk_overlap = 50,
            )
    chunks=[]
    for doc in documents:
        for chunk in split_to_chunk.split_text(clean_text_for_chunking(doc)):
            chunks.append(chunk)
    return chunks

In [38]:
text = re.sub(r'\n{2,}', '[[PARAGRAPH]]', doc)

In [39]:
chunks = document_to_chunks([doc])

In [44]:
_=[print(i,chunk) for i,chunk in enumerate(chunks) if 'We may note' in chunk]

28 Thus the first stage of aphoristic or unconscious morality is shown to be inadequate to the wants of the age; the authority of the poets is set aside, and through the winding mazes of dialectic we make an approach to the Christian precept of forgiveness of injuries. Similar words are applied by the Persian mystic poet to the Divine being when the questioning spirit is stirred within him:—‘If because I do evil, Thou punishest me by evil, what is the difference between Thee and me?’ In this both Plato and Kheyam rise above the level of many Christian (?) theologians. The first definition of justice easily passes into the second; for the simple words ‘to speak the truth and pay your debts’ is substituted the more abstract ‘to do good to your friends and harm to your enemies.’ Either of these explanations gives a sufficient rule of life for plain men, but they both fall short of the precision of philosophy. We may note in passing the antiquity of casuistry, which not only arises out of 

In [4]:
corpus={}
for i,chunk in enumerate(chunks):
    corpus[i]=chunk

In [5]:
from datasets import load_dataset
import random

In [6]:
corpus2 = load_dataset("BeIR/webis-touche2020", "corpus", split="corpus")
queries2 = load_dataset("BeIR/webis-touche2020", "queries", split="queries")
relevant_docs_data2 = load_dataset("BeIR/webis-touche2020-qrels", split="test")

In [7]:
corpus2 = corpus2.map(lambda x: {'text': x['title'] + " " + x['text']}, remove_columns=['title'])

# Shrink the corpus size heavily to only the relevant documents + 30,000 random documents
required_corpus_ids = set(map(str, relevant_docs_data2["corpus-id"]))
required_corpus_ids |= set(random.sample(corpus2["_id"], k=10_000))
corpus2 = corpus2.filter(lambda x: x["_id"] in required_corpus_ids)

# Convert the datasets to dictionaries
corpus2 = dict(zip(corpus2["_id"], corpus2["text"]))  # Our corpus (cid => document)

Filter:   0%|          | 0/382545 [00:00<?, ? examples/s]

In [8]:
corpus_n=max(list(corpus.keys()))+1

In [9]:
corpus_all=corpus.copy()
for i,key in enumerate(corpus2.keys()):
    corpus_all[corpus_n+i]=corpus2[key]

In [10]:
print(max(list(corpus.keys())),max(list(corpus_all.keys())))

379 12425


In [11]:
import pickle

with open("../data/corpus_all.pkl","wb") as f:
    pickle.dump(corpus_all,f)

In [15]:
import pickle

with open("../data/corpus_all.pkl","rb") as f:
    corpus=pickle.load(f)

In [15]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

In [12]:
from qdrant_client import QdrantClient

def get_qdrant_client():
    """Create a singleton Qdrant client."""
    return QdrantClient("http://localhost:6333")

In [13]:
from typing import Dict, List
collection_name='simple_rag'

def semantic_search(
    query: str, collection_name: str, top_k: int = 5
) -> List[Dict]:
    """Perform semantic search on code chunks."""
    contexts=[]
    qry_vec = model.encode(query).tolist() 
    client = get_qdrant_client()

    try:
        results = client.query_points(
            collection_name=collection_name, query=qry_vec, limit=top_k
        )
        return [
            {
                "id": hit.id,
                "file_path": hit.payload["file_path"],
                "book": hit.payload["book"],
                "text": hit.payload["text"],
                "score": hit.score,
            }
            for hit in results.points
        ]
    except Exception as e:
        print(f"Search error: {e}")
        return []

In [17]:
import numpy as np
norm = np.linalg.norm
def cosine_distance(v1,v2):
    v1=np.array(v1)
    v2=np.array(v2)
    return v1.dot(v2)/norm(v1)/norm(v2)

In [3]:
df=pd.read_csv('../data/cleanse_q_a.csv')

In [4]:
queries=df["question"].to_dict()

In [12]:
results=[]
for k in queries.keys():
    result=semantic_search(queries[k],collection_name=collection_name)
    text_ids=[i["id"] for i in result]
    #print(text_ids)
    results.append([k,text_ids])

In [18]:
similarities_results=[]
for r in results:
    q_vec=model.encode(queries[r[0]]).tolist()
    #print(r[0],q_vec)
    similarities=[]
    for t in r[1]:
        t_vec=model.encode(corpus[t]).tolist()
        sim=cosine_distance(q_vec,t_vec)
        if sim > 0.7:
            similarities.append([t,sim])
    similarities_results.append([r[0],similarities])

In [24]:
relevant_docs={}
for res in similarities_results:
    texts=[]
    for doc in res[1]:
        texts.append(doc[0])
    relevant_docs[res[0]]=texts
#relevant_doc

In [7]:
import pickle

with open("../data/queries.pkl","wb") as f:
    pickle.dump(queries,f)

In [ ]:
with open("../data/relevant_docs.pkl","wb") as f:
    pickle.dump(relevant_docs,f)

In [6]:
queries

{0: 'What does Plato consider as the scope of education?',
 1: "How is Plato's view on education unique compared to other writers?",
 2: 'In which work does Plato discuss his views on education?',
 3: "What makes Plato's ideas about education applicable to modern life?",
 4: 'What did mathematics represent to Plato compared to its current scope?',
 5: 'How significant was the role of mathematics within human knowledge according to Plato?',
 6: 'Why were mathematical concepts considered inexhaustible by Plato?',
 7: 'According to Plato, what was the ultimate ground of mathematical ideas?',
 8: 'What higher conception of knowledge did Plato consider ideas of number to become secondary to?',
 9: 'What does Plato argue about the nature of evil?',
 10: 'How does the orrery of the heavens in the Republic differ from that in the Timaeus?',
 11: 'What does Plato teach about education?',
 12: 'Would Plato ever allow any kind of education to cease?',
 13: 'What is the main distinction between th

In [9]:
import pickle

with open("../data/corpus_all.pkl","rb") as f:
    corpus_all=pickle.load(f)

In [17]:
semantic_search("how do we feel about Plato critization about the practice of medicine in his day?",collection_name=collection_name)

[{'id': 334,
  'file_path': 'data/the_republic_introduction.txt',
  'book': 'the republic',
  'text': '\n\nThe perplexity of medicine is paralleled by the perplexity of law; in\nwhich, again, Plato would have men follow the golden rule of\nsimplicity. Greater matters are to be determined by the legislator or\nby the oracle of Delphi, lesser matters are to be left to the temporary\nregulation of the citizens themselves. Plato is aware that laissez\nfaire is an important element of government. The diseases of a State\nare like the heads of a hydra; they multiply when they are cut off. The\ntrue remedy for them is not extirpation but prevention. And the way to\nprevent them is to take care of education, and education will take care\nof all the rest. So in modern times men have often felt that the only\npolitical measure worth having—the only one which would produce any\ncertain or lasting effect, was a measure of national education. And in\nour own more than in any previous age the necess

In [10]:
for id in [234, 267, 244, 78]:
    print(id,corpus_all[id])

234 

Twice has the just man overthrown the unjust—once more, as in an
Olympian contest, first offering up a prayer to the saviour Zeus, let
him try a fall. A wise man whispers to me that the pleasures of the
wise are true and pure; all others are a shadow only. Let us examine
this: Is not pleasure opposed to pain, and is there not a mean state
which is neither? When a man is sick, nothing is more pleasant to him
than health. But this he never found out while he was well. In pain he
desires only to cease from pain; on the other hand, when he is in an
ecstasy of pleasure, rest is painful to him. Thus rest or cessation is
both pleasure and pain. But can that which is neither become both?
Again, pleasure and pain are motions, and the absence of them is rest;
but if so, how can the absence of either of them be the other? Thus we
are led to infer that the contradiction is an appearance only, and
witchery of the senses. And these are not the only pleasures, for there
are others which have no